In [1]:
# --- 1. ライブラリのインポート ---
import pandas as pd
import joblib
import MeCab
import ipadic
import re  # 正規表現モジュールを追加
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# --- 2. 日本語前処理関数の定義 (学習時と同じものを定義) ---
# ★ここが修正ポイントです★

# グローバル変数として MeCab Tagger を準備
chaser = MeCab.Tagger(ipadic.MECAB_ARGS)

def japanese_tokenizer(text):
    """
    日本語の文章を単語リストに変換する関数
    (学習時の定義と完全に一致させる必要があります)
    """
    text = str(text)
    
    # URLを <URL> という文字列に置換
    text = re.sub(r'https?://[\w/:%#\$&\?\(\)~\.=\+\-]+', '<URL>', text)
    
    parsed = chaser.parse(text)
    words = []
    
    for line in parsed.split('\n'):
        if line == 'EOS' or line == '':
            continue
            
        parts = line.split('\t')
        word_surface = parts[0]
        features = parts[1].split(',')
        
        pos = features[0] # 品詞
        target_pos = ['名詞', '動詞', '形容詞', '接頭詞']
        
        if pos in target_pos:
            # 不要な名詞を除外 (非自立, 代名詞, 数)
            if features[1] in ['非自立', '代名詞', '数']:
                continue
                
            # 原形があればそれを使う
            if len(features) > 6 and features[6] != '*':
                word = features[6]
            else:
                word = word_surface
                
            words.append(word)
            
    return words

# --- 3. データの読み込みと前処理（データセット再現） ---
print("データを読み込み中...")
csv_path = 'dataset/narou_dataset.csv'
df = pd.read_csv(csv_path)

# 欠損値埋め
df['あらすじ'] = df['あらすじ'].fillna('')

# ==========================================
# 分析対象のジャンルID
TARGET_GENRE_ID = None 

genres_map = {
    0: '未設定', 101: '異世界（恋愛）', 102: '現実世界（恋愛）',
    201: 'ハイファンタジー', 202: 'ローファンタジー',
    301: '純文学', 302: 'ヒューマンドラマ', 303: '歴史',
    304: '推理', 305: 'ホラー', 306: 'アクション', 307: 'コメディー',
    401: 'VRゲーム', 402: '宇宙', 403: '空想科学', 404: 'パニック',
    9901: '童話', 9902: '詩', 9903: 'エッセイ', 9904: 'リプレイ',
    9999: 'その他', 9801: 'ノンジャンル'
}
genre_name = genres_map.get(TARGET_GENRE_ID, 'all')

# 1. 文字数によるフィルタリング
df_filtered = df.copy()

# 2. ジャンルで絞り込み
if TARGET_GENRE_ID is not None:
    df_genre = df_filtered[df_filtered['作品ジャンル'] == TARGET_GENRE_ID].copy()
else:
    df_genre = df_filtered.copy()

# 3. エタる群(1)と完結群(0)の均衡化
df_eternal = df_genre[df_genre['is_eternal'] == 1]
df_complete = df_genre[df_genre['is_eternal'] == 0]

min_count = min(len(df_eternal), len(df_complete))

df_eternal_sampled = df_eternal.sample(n=min_count, random_state=42)
df_complete_sampled = df_complete.sample(n=min_count, random_state=42)

df_balanced = pd.concat([df_eternal_sampled, df_complete_sampled]).sample(frac=1, random_state=42)

# 学習データ作成
X_text = df_balanced['あらすじ'].values
y = df_balanced['is_eternal'].values

print("-" * 30)
print(f"データセット再現完了: {len(df_balanced)}件")

# --- 4. テストデータの分離 ---
X_train, X_test, y_train, y_test = train_test_split(
    X_text, y, test_size=0.2, random_state=42, stratify=y
)
print(f"テストデータ数: {len(X_test)}件")

# --- 5. 学習済みモデルの読み込みと評価 ---
model_path = f'result_tfidf/model_{genre_name}.pkl' 

try:
    print(f"モデルを読み込み中: {model_path}")
    
    # ここでエラーが出なくなります（関数が定義されているため）
    loaded_model = joblib.load(model_path)
    
    # 予測を実行
    print("予測を実行中...")
    y_pred = loaded_model.predict(X_test)

    print("\n" + "="*40)
    print(f"【評価結果】 ジャンル: {genre_name}")
    print("="*40)
    
    # ★ここに適合率・再現率・F1値が表示されます★
    report = classification_report(y_test, y_pred, target_names=['完結(0)', 'エタり(1)'])
    print(report)
    
    print("\n--- 混同行列 ---")
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    print(f"TN(正しく完結): {tn}")
    print(f"FP(完結なのにエタり): {fp}")
    print(f"FN(エタりなのに完結): {fn}")
    print(f"TP(正しくエタり): {tp}")

except FileNotFoundError:
    print(f"エラー: モデルファイルが見つかりません: {model_path}")
except Exception as e:
    print(f"予期せぬエラーが発生しました: {e}")

データを読み込み中...
------------------------------
データセット再現完了: 292486件
テストデータ数: 58498件
モデルを読み込み中: result_tfidf/model_all.pkl
予測を実行中...

【評価結果】 ジャンル: all
              precision    recall  f1-score   support

       完結(0)       0.68      0.72      0.70     29249
      エタり(1)       0.70      0.66      0.68     29249

    accuracy                           0.69     58498
   macro avg       0.69      0.69      0.69     58498
weighted avg       0.69      0.69      0.69     58498


--- 混同行列 ---
TN(正しく完結): 21150
FP(完結なのにエタり): 8099
FN(エタりなのに完結): 9925
TP(正しくエタり): 19324
